# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading and exploring the [FAIR^2 dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://github.com/mlcommons/croissant), accessible via a remote URL.

In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`. This notebook will point to the FAIR^2 schema directly from its URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL to the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview

Explore available record sets, fields, and their `@id`. All access to data elements will reference their Croissant `@id` for precision and reproducibility. We will enumerate the dataset's structure so users can select relevant elements for downstream tasks.

In [ ]:
# List all available record sets and their fields, using @id
for record_set in metadata.record_sets:
    print(f"RecordSet: {record_set['@id']}")
    if 'field' in record_set:
        for field in record_set['field']:
            # Field is usually a dict or reference (shortcut), so handle both
            if isinstance(field, dict):
                print(f"  Field: {field['@id']}")
            elif isinstance(field, str):
                print(f"  Field: {field}")
    print()

## 3. Data Extraction

Now, let's extract data from one or multiple record sets. Specify a list of record set `@id`s discovered above, and load them into pandas DataFrames for analysis.

In [ ]:
# Choose which record sets to extract
# (Replace the example IDs below with the actual ones from the previous step.)
record_set_ids = [
    '@id:cr:RecordSet:protein_abundance',
    '@id:cr:RecordSet:peptide_details',
    # ...add more if needed
]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from {record_set_id}.")
        print(f"Fields: {df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

Let's process the primary record set (e.g., `@id:cr:RecordSet:protein_abundance`) by filtering, normalizing, and grouping numeric fields. **Be sure to reference fields using their `@id`.**

In [ ]:
# Example field IDs (replace with actual field IDs found in your dataset)
record_set_id = '@id:cr:RecordSet:protein_abundance'
numeric_field = '@id:cr:Field:abundance'
group_field = '@id:cr:Field:protein_group_label'

df = dataframes.get(record_set_id)
if df is not None and numeric_field in df.columns:
    # Remove non-numerics in the numeric field, coerce to float
    num = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = num.mean()  # Example threshold: above average
    filtered_df = df[num > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.2f}, n={len(filtered_df)}")

    # Normalize
    filtered_num = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
    filtered_df[f"{numeric_field}_normalized"] = (filtered_num - filtered_num.mean()) / filtered_num.std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped by {group_field}, mean {numeric_field}:")
        print(grouped_df.head())
else:
    print(f"DataFrame for {record_set_id} missing or {numeric_field} not found.")

## 5. Visualization

Visualize the distribution of a numeric variable, or any relationship of interest. Here we show a histogram and (if group_field is present) a barplot of grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    pd.to_numeric(df[numeric_field], errors='coerce').hist(bins=30)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Barplot by group
    if group_field in df.columns:
        group_means = df.groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(10,5))
        sns.barplot(x=group_field, y=numeric_field, data=group_means)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrates how to explore and process a dataset described by a Croissant schema using the `mlcroissant` library. By referencing all entities via their Croissant `@id`, analysis steps are precise and easily reproducible. Typical data science workflows (data loading, overview, EDA, normalization, grouping, and visualization) are supported natively.

Replace the example `@id`s with actual values printed in the Data Overview section for your dataset exploration. See the [mlcroissant documentation](https://github.com/mlcommons/croissant) for further operations and API details.